# Phase 5: matched development evaluation

**Research question:** Can a learned return ranker improve stock ordering, and can a learned risk classifier improve Low/Medium/High volatility prediction on later chronological blocks?

Four outer development folds use the frozen Phase 4 protocol. Return is a same-date peer class; risk classes use cutoffs computed from each fold's training outcomes. All signals use information available through session *t* and are available after that close. The 2024–2025 reused test period is excluded from selection.

This notebook is arranged for Google Colab. Cell 1 clones the selected Git branch and installs a Colab-compatible scientific Python stack. After its first run, restart the Colab session if package versions changed, then run all cells again. Cell 2 downloads the Git-ignored Phase 3 dataset and Phase 4 protocol from the project's shared Google Drive folder. The source package verifies their frozen checksums, fits any missing Phase 5 model artifacts, and writes tables and SVG figures under `reports/generated/current/model_evaluation_package/`. The model fits can take several minutes.

In [ ]:
from pathlib import Path
import sys

REPOSITORY = Path("/content/HalalifyAI")
BRANCH = "feature/model-evaluation"

# Clone the source code. Large generated research artifacts are copied in Cell 2.
if not REPOSITORY.is_dir():
    !git clone --branch {BRANCH} --single-branch \
        https://github.com/frager098/HalalifyAI.git \
        /content/HalalifyAI

%cd /content/HalalifyAI

# Install Colab-compatible dependencies required by this notebook.
%pip install --no-cache-dir --force-reinstall -q \
    "numpy==2.5.3" \
    "pandas==3.0.6" \
    "scipy==1.18.1" \
    "scikit-learn==1.9.1" \
    "exchange-calendars==4.13.2" \
    "joblib==1.6.0" \
    "threadpoolctl==3.7.0" \
    "gdown>=5.2,<6"
root = REPOSITORY.resolve()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Project root:", root)
print("Branch:")
!git branch --show-current

/content/HalalifyAI
Project root: /content/HalalifyAI
Branch:
feature/model-evaluation


In [1]:
import gdown
import shutil

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1WQNAY14GOl1zznaXXCmF91mjIn-vHQiP"
download_dir = Path("/content/HalalifyAI-artifacts")
download_dir.mkdir(parents=True, exist_ok=True)

downloaded = gdown.download_folder(
    url=DRIVE_FOLDER_URL,
    output=str(download_dir),
    quiet=False,
    use_cookies=False,
)
if not downloaded:
    raise RuntimeError(
        "No files were downloaded. Confirm that the Google Drive folder is shared "
        "with anyone who has the link."
    )

artifacts = {
    "market_features_and_labels.csv": (
        root / "data/processed/model_inputs/market_features_and_labels_v1/market_features_and_labels.csv"
    ),
    "chronological_evaluation_protocol.json": (
        root / "reports/generated/current/chronological_evaluation_protocol.json"
    ),
}

for filename, destination in artifacts.items():
    matches = list(download_dir.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename!r} in the shared folder; found {len(matches)}."
        )
    source = matches[0]
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print(f"Copied: {source.name} -> {destination}")

NameError: name 'Path' is not defined

In [ ]:
# Check an already-loaded NumPy before importing SciPy or scikit-learn.
# Replacing NumPy with pip cannot replace the copy held in kernel memory.
loaded_numpy = sys.modules.get("numpy")
if loaded_numpy is not None and tuple(map(int, loaded_numpy.__version__.split(".")[:2])) >= (2, 3):
    raise RuntimeError(
        f"Colab still has NumPy {loaded_numpy.__version__} loaded in memory. "
        "Select Runtime > Restart session, then run all cells again."
    )

import numpy as np
import pandas as pd
import scipy
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

In [3]:
from src.model_training.build_model_evaluation_report import (
    build_results_package,
    show_svg,
    show_table,
)

summary, tables, artifact_dir = build_results_package()
print("Research rows:", summary["data"]["rows"], "companies:", summary["data"]["companies"])
print("Return eligible:", summary["data"]["return_eligible"], "Risk eligible:", summary["data"]["risk_eligible"])
print("Figures and tables:", artifact_dir)

AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

## Cohort and fold boundaries

Each fold uses shared market session dates. Training outcomes end before evaluation starts; the risk cutoffs shown here are computed from outer training outcomes only. Model input columns come from the frozen feature groups and exclude labels, target dates, identifiers, and provenance.

In [ ]:
folds = tables['folds.csv'].copy()
show_table(folds.round({'risk_low_cutoff': 4, 'risk_high_cutoff': 4}))

## Return: same-date stock ordering

The learned score is `P(Higher) - P(Lower)`; it is an ordering score, **not** a percentage return. The training-class prior is constant across stocks on a date, so its rank correlation and top-10 selection are undefined. Trailing 20-day momentum is a simple rule-based benchmark, not a fitted model. Rank correlation is the primary return metric. Top-10 forward-close spreads are research selection diagnostics.

In [ ]:
return_metrics = tables['return_fold_metrics.csv'].copy()
show_table(return_metrics.round({'mean_daily_spearman': 4, 'mean_top10_minus_universe': 4, 'macro_f1': 4}))
show_svg(artifact_dir / 'return_fold_spearman.svg')
print('Equal-weight mean fold Spearman:', {name: round(value, 4) for name, value in summary['return_mean_fold_spearman'].items()})

**Return decision for this development review:** No return approach is selected. Momentum's mean rank correlation is effectively zero. Logistic Regression's higher mean varies across folds, and the boosted tree has almost no mean ordering signal. A new or revised approach requires a separately frozen evaluation before a final generalization claim.

## Risk: class quality and severe misses

Macro F1 is the primary metric. High recall and **High predicted Low** reveal risk underestimation that the average can hide. Class prior and trailing volatility use the same outer rows and fold-specific risk boundaries as the learned models. Log loss and multiclass Brier apply only where probabilities exist.

In [ ]:
risk_metrics = tables['risk_fold_metrics.csv'].copy()
show_table(risk_metrics.round({'macro_f1': 4, 'high_recall': 4, 'ordinal_mae': 4, 'log_loss': 4, 'multiclass_brier': 4}))
show_svg(artifact_dir / 'risk_fold_f1.svg')
risk_summary = pd.DataFrame(summary['risk_summary']).T
show_table(risk_summary[['mean_fold_macro_f1', 'mean_fold_high_recall', 'total_high_predicted_low', 'pooled_high_predicted_low_rate', 'mean_fold_ordinal_mae']].round(4).reset_index(names='method'))

### Per-class errors and confusion matrices

Rows of each confusion matrix are observed risk classes; columns are predicted classes. All four folds appear below so failure cases remain visible beside average scores.

In [ ]:
risk_classes = tables['risk_class_metrics.csv']
show_table(risk_classes[risk_classes.method.isin(['LogisticRegression', 'HistGradientBoosting'])].round(4))
confusion = tables['risk_confusion_matrices.csv']
for fold_number in range(1, 5):
    for method in ('LogisticRegression', 'HistGradientBoosting'):
        subset = confusion[(confusion.fold == fold_number) & (confusion.method == method)]
        matrix = subset.pivot(index='observed', columns='predicted', values='count').reindex(index=['Low', 'Medium', 'High'], columns=['Low', 'Medium', 'High'])
        print(f'Fold {fold_number}: {method}; observed rows, predicted columns')
        show_table(matrix.reset_index())

**Phase 5 risk conclusion:** HistGradientBoosting is the research candidate for later validation. It beats trailing volatility on macro F1 in all four folds, but its mean advantage over Logistic Regression is small. Trailing volatility has slightly higher High recall and 25 fewer High-to-Low misses across 20,502 High outcomes. An acceptable safety tradeoff has not been approved, so this candidate is not a user-facing risk recommendation. Numerical MAE/RMSE/R² in the risk review apply only to the direct trailing-volatility forecast; classifier probabilities are not numerical volatility forecasts.

## Reproducibility and limits

The summary lists exact dataset, protocol, model-prediction and artifact hashes plus library versions. The source code rejects mismatched saved fits. Generated files are intentionally Git-ignored; source modules, this notebook, and `decision.md` are reviewable in Git. This is an academic research comparison, not a compliant portfolio or executable backtest.

In [ ]:
print('Dataset SHA-256:', summary['dataset_sha256'])
print('Protocol SHA-256:', summary['protocol_sha256'])
print('Environment:', summary['environment'])
print('Prediction hashes:', summary['model_prediction_sha256'])
print('Artifact hashes:', summary['artifacts'])
print('Limitations:')
for item in summary['limitations']:
    print('•', item)